In [10]:
from modelo import cargar_instancia, crear_modelo_convrp
import gurobipy as gp
from gurobipy import GRB
import pandas as pd
from IPython.display import display

# Modelos

In [11]:
def optimizar_modelo_base(datos):
    modelo = crear_modelo_convrp(datos)
    
    modelo.Params.Threads = 1
    modelo.Params.TimeLimit = 600
    
    print("¡Optimizando modelo base!\n")
    modelo.optimize()
    return modelo

def optimizar_modelo_relajado(datos):
    modelo = crear_modelo_convrp(datos)
    
    # Convierte las variables binarias a continuas
    modelo = modelo.relax()
    
    modelo.Params.Threads = 1
    modelo.Params.TimeLimit = 600
    
    print("¡Optimizando modelo relajado!\n")
    modelo.optimize()
    return modelo

def optimizar_modelo_cuts(datos):
    modelo = crear_modelo_convrp(datos)
    
    modelo.Params.Threads = 1
    modelo.Params.TimeLimit = 600
    modelo.Params.Cuts = 0
    
    print("¡Optimizando modelo con cuts = 0!\n")
    modelo.optimize()
    return modelo

def optimizar_modelo_presolve(datos):
    modelo = crear_modelo_convrp(datos)
    
    modelo.Params.Threads = 1
    modelo.Params.TimeLimit = 600
    modelo.Params.Presolve = 0
    
    print("¡Optimizando modelo con presolve = 0!\n")
    modelo.optimize()
    return modelo

def optimizar_modelo_mipfocus_1(datos):
    modelo = crear_modelo_convrp(datos)
    
    modelo.Params.Threads = 1
    modelo.Params.TimeLimit = 600
    modelo.Params.MIPFocus = 1
    
    print("¡Optimizando modelo con MIPFocus = 1!\n")
    modelo.optimize()
    return modelo

def optimizar_modelo_mipfocus_3(datos):
    modelo = crear_modelo_convrp(datos)
    
    modelo.Params.Threads = 1
    modelo.Params.TimeLimit = 600
    modelo.Params.MIPFocus = 3
    
    print("¡Optimizando modelo con MIPFocus = 3!\n")
    modelo.optimize()
    return modelo

def optimizar_modelo_heuristics(datos):
    modelo = crear_modelo_convrp(datos)
    
    modelo.Params.Threads = 1
    modelo.Params.TimeLimit = 600
    modelo.Params.Heuristics = 0
    
    print("¡Optimizando modelo con heuristics = 0!\n")
    modelo.optimize()
    return modelo

def optimizar_modelo_tuneado(datos, nombre_instancia):
    modelo = crear_modelo_convrp(datos)
    
    modelo.Params.Threads = 1
    modelo.Params.TimeLimit = 600
    modelo.Params.TuneTimeLimit = 18000
    
    print("¡Optimizando parametros!\n")
    modelo.tune()
    
    if modelo.TuneResultCount > 0:
        modelo.getTuneResult(0)
        modelo.write(f"parametros/mejores_parametros_{nombre_instancia}.prm")
    
    print("¡Optimizando modelo con parametros tuneados!")
    modelo.optimize()
    return modelo

# Funciones para análisis de resultados

In [ ]:
def reporte_modelo(modelo, nombre_modelo, nombre_instancia):
    tiempo_ejecucion = modelo.Runtime
    
    if modelo.SolCount > 0:
        valor_objetivo = modelo.ObjVal
        gap_final = modelo.MIPGap
    else:
        valor_objetivo = None
        gap_final = None
        
    print(f"¡Reporte {nombre_modelo} {nombre_instancia}!\n"
          f"Tiempo de ejecución: {tiempo_ejecucion:.2f}\n"
          f"Valor objetivo: {valor_objetivo}\n"
          f"Gap final: {gap_final}\n")

def comparacion_baseline_relajado(baseline, relajado):
    if baseline.SolCount > 0 and relajado.SolCount > 0:
        brecha_integralidad = 100 * (baseline.ObjVal - relajado.ObjVal) / baseline.ObjVal
        print(f"Valor objetivo modelo baseline: {baseline.ObjVal:2f}\n"
            f"Valor objetivo modelo relajado: {relajado.ObjVal:2f}\n"
            f"Brecha de integralidad: {brecha_integralidad}\n")
    else:
        print("No se puede calcular la brecha de integralidad puesto que nos e obtuvo una solucion factible en alguno de los modelos.\n")

def comparacion_modelos(modelos, nombres, instancia):
    data = {
            "Modelo": [],
            "Tiempo": [],
            "Valor Objetivo": [],
            "Gap Final": []
            }
    
    for i, m in enumerate(modelos):
        data["Modelo"].append(nombres[i])
        data["Tiempo"].append(m.Runtime)
        data["Valor Objetivo"].append(m.ObjVal)
        data["Gap Final"].append(m.MIPGap)
    
    print(f"=== Comparativa de modelos {instancia} ===\n")
    df = pd.DataFrame(data)
    display(df)
    
def analisis_sensibilidad_capacidad(modelo, nombre_instancia, tol=1e-6):
    if modelo.SolCount == 0:
        print(f"[{nombre_instancia}] No hay solución incumbente: no se puede fijar el modelo.")
        return None

    fm = modelo.fixed()          # MIP -> LP con enteras fijas en la incumbente
    fm.Params.Presolve = 0       # evita que se eliminen filas con solo variables fijas
    fm.Params.Method = 1         # simplex dual
    fm.Params.Threads = 1
    fm.Params.OutputFlag = 0
    fm.optimize()

    if fm.Status != GRB.OPTIMAL:
        print(f"[{nombre_instancia}] El modelo fijado no es óptimo (estado {fm.Status}).")
        return None

    filas = []
    for c in fm.getConstrs():
        if c.ConstrName.startswith("Capacidad"):
            carga = c.RHS - c.Slack
            filas.append({
                "Restricción": c.ConstrName,
                "Q (RHS)": c.RHS,
                "Carga": carga,
                "Holgura": c.Slack,
                "Activa": c.Slack < tol,
                "Precio sombra (Pi)": c.Pi,
                "SARHSLow": c.SARHSLow,
                "SARHSUp": c.SARHSUp,
            })

    df = pd.DataFrame(filas)

    print(f"=== Sensibilidad de Q, instancia {nombre_instancia} ===")
    print(f"Objetivo MIP: {modelo.ObjVal:.4f} | Objetivo LP fijado: {fm.ObjVal:.4f}")
    print(f"Restricciones de capacidad: {len(df)} | activas: {int(df['Activa'].sum())}")
    print(f"Carga máxima observada: {df['Carga'].max():.2f} (Q = {df['Q (RHS)'].iloc[0]:.2f})")
    print(f"Filas con precio sombra distinto de 0: {(df['Precio sombra (Pi)'].abs() > tol).sum()}")
    display(df.sort_values("Carga", ascending=False).head(15))
    return df, fm

In [13]:
datos_1 = cargar_instancia("instancias/instancia_1.json")
datos_2 = cargar_instancia("instancias/instancia_2.json")

## Modelo Base

In [18]:
modelo_base_I1 = optimizar_modelo_base(datos_1)
reporte_modelo(modelo_base_I1, "modelo base", "I1")

Set parameter Threads to value 1
Set parameter TimeLimit to value 600
¡Optimizando modelo base!

Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26300.2))

CPU model: 11th Gen Intel(R) Core(TM) i7-11800H @ 2.30GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 8 physical cores, 16 logical processors, using up to 1 threads

Non-default parameters:
TimeLimit  600
Threads  1

Optimize a model with 2316 rows, 2208 columns and 12690 nonzeros (Min)
Model fingerprint: 0xf3811d06
Model has 2160 linear objective coefficients
Variable types: 48 continuous, 2160 integer (2160 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+04]
  Objective range  [1e+00, 4e+01]
  Bounds range     [1e+00, 2e+03]
  RHS range        [1e+00, 1e+04]

Presolve removed 1222 rows and 1166 columns
Presolve time: 0.34s
Presolved: 1094 rows, 1042 columns, 6481 nonzeros
Variable types: 31 continuous, 1011 integer (1011 binary)

Root relaxation: objective 1.081800e+02, 255 iteratio

In [ ]:
modelo_base_I2 = optimizar_modelo_base(datos_2)
reporte_modelo(modelo_base_I2, "modelo base", "I2")

## Modelo Relajado

In [ ]:
modelo_relajado_I1 = optimizar_modelo_relajado(datos_1)
reporte_modelo(modelo_relajado_I1, "modelo relajado", "I1")

# Comparacion del baseline con el relajado
comparacion_baseline_relajado(modelo_base_I1, modelo_relajado_I1)

In [ ]:
modelo_relajado_I2 = optimizar_modelo_relajado(datos_2)
reporte_modelo(modelo_relajado_I2, "modelo relajado", "I2")

# Comparacion del baseline con el relajado
comparacion_baseline_relajado(modelo_base_I2, modelo_relajado_I2)

## Prueba de configuración de modelos
### Modelo Cuts = 0

In [ ]:
# Optimización del modelo con configuración 1 con ambas instancias
modelo_cuts_I1 = optimizar_modelo_cuts(datos_1)
reporte_modelo(modelo_cuts_I1, "modelo Cuts = 0", "I1")

In [ ]:
modelo_cuts_I2 = optimizar_modelo_cuts(datos_2)
reporte_modelo(modelo_cuts_I2, "modelo Cuts = 0", "I2")

### Modelo Presolve = 0

In [ ]:
# Optimización del modelo con configuración 2 con ambas instancias
modelo_presolve_I1 = optimizar_modelo_presolve(datos_1)
reporte_modelo(modelo_presolve_I1, "modelo Presolve = 0", "I1")


In [ ]:
modelo_presolve_I2 = optimizar_modelo_presolve(datos_2)
reporte_modelo(modelo_presolve_I2, "modelo Presolve = 0", "I2")

### Modelo MIPFocus = 1

In [ ]:
modelo_mipfocus1_I1 = optimizar_modelo_mipfocus_1(datos_1)
reporte_modelo(modelo_mipfocus1_I1, "modelo MIPFocus = 1", "I1")

In [ ]:
modelo_mipfocus1_I2 = optimizar_modelo_presolve(datos_2)
reporte_modelo(modelo_mipfocus1_I2, "modelo MIPFocus = 1", "I2")

### Modelo MIPFocus = 3

In [ ]:
modelo_mipfocus3_I1 = optimizar_modelo_mipfocus_3(datos_1)
reporte_modelo(modelo_mipfocus3_I1, "modelo MIPFocus = 3", "I1")

In [ ]:
modelo_mipfocus3_I2 = optimizar_modelo_mipfocus_3(datos_2)
reporte_modelo(modelo_mipfocus3_I2, "modelo MIPFocus = 3", "I2")

### Modelo Heuristics = 0

In [ ]:
modelo_heuristics_I1 = optimizar_modelo_heuristics(datos_1)
reporte_modelo(modelo_heuristics_I1, "modelo Heuristics = 0", "I1")

In [ ]:
modelo_heuristics_I2 = optimizar_modelo_heuristics(datos_2)
reporte_modelo(modelo_heuristics_I2, "modelo Heuristics = 0", "I2")

### Modelo tuneado

In [ ]:
modelo_tuneado_I1 = optimizar_modelo_tuneado(datos_1, "I1")
reporte_modelo(modelo_tuneado_I1, "modelo tuneado", "I1")

In [ ]:
modelo_tuneado_I2 = optimizar_modelo_tuneado(datos_2, "I2")
reporte_modelo(modelo_tuneado_I2, "modelo tuneado", "I2")

# Comparativa de modelos

In [ ]:
modelos_I1 = [modelo_base_I1, modelo_relajado_I1, modelo_cuts_I1, modelo_presolve_I1, modelo_mipfocus1_I1, modelo_mipfocus3_I1, modelo_heuristics_I1, modelo_tuneado_I1]

nombres = ["Baseline", "Relajado", "Cuts=0", "Presolve=0", "MIPF=1", "MIPF=3", "Heuristics=0","Tuneado"]

comparacion_modelos(modelos_I1, nombres, "I1")

In [ ]:
modelos_I2 = [modelo_base_I2, modelo_relajado_I2, modelo_cuts_I2, modelo_presolve_I2, modelo_mipfocus1_I2, modelo_mipfocus3_I2, modelo_heuristics_I2, modelo_tuneado_I2]

comparacion_modelos(modelos_I2, nombres, "I2")

#  Analisis de sensibilidad

In [ ]:
res_sens_I1 = analisis_sensibilidad_capacidad(modelo_tuneado_I1, "I1")

In [ ]:
res_sens_I2 = analisis_sensibilidad_capacidad(modelo_tuneado_I2, "I2")